In [2]:
import torch
from torch import nn
from torch.nn import L1Loss as MAE
import numpy as np

In [137]:
def get_roi_indexs(x:torch.Tensor, axes=None, square=False):

    with torch.no_grad():
        device=x.device
        x=torch.split(x,1,dim=1)
        indexs=[]
        for c in range(0,len(x)):
            temp=x[c].flatten()
            label=temp.unique()
            for l in label:
                if l ==0: 
                    continue
                where=(temp == l).nonzero().flatten().to(device)
                indexs.append(where)

    return indexs

class value_DVH_Loss(nn.Module):
    def __init__(self, ddp: bool = True):
        """
        """
        super(value_DVH_Loss, self).__init__()

        self.ddp = ddp

    def forward(self, x, y, mask_index):
        with torch.no_grad():
            x=x.flatten()
            y=y.flatten()
            maes=torch.tensor(0)
            L=len(mask_index)
            
            for l in mask_index:

                dose_x=torch.take(x,l)
                dose_y=torch.take(y,l) 
                                
                dose_x,_=torch.sort(dose_x)
                dose_y,_=torch.sort(dose_y) 
                
                maes=maes+torch.mean(torch.abs(dose_x-dose_y))
        return maes/L


In [1]:
l=torch.randint(0,3,(2,1,2,2))
x=torch.randint(10,12,(2,1,2,2))
y=torch.randint(10,12,(2,1,2,2))


NameError: name 'torch' is not defined

In [145]:
l

tensor([[[[1, 2],
          [2, 0]]],


        [[[0, 1],
          [2, 2]]]])

In [146]:
x

tensor([[[[10, 10],
          [11, 11]]],


        [[[10, 11],
          [11, 10]]]])

In [147]:
y

tensor([[[[10, 10],
          [11, 10]]],


        [[[11, 11],
          [11, 11]]]])

In [149]:
indexs=get_roi_indexs(l)
print(indexs)

[tensor([0, 5]), tensor([1, 2, 6, 7])]


In [151]:
DVH = value_DVH_Loss(True)
dvh_loss = DVH(x, y, mask_index=indexs)
print(dvh_loss)

tensor(0.1250)


In [8]:
l=torch.rand((3,))
l

tensor([0.3564, 0.2031, 0.3455])

In [11]:
l.quantile(0.1)

tensor(0.2316)